In [ ]:
%pip install -U -qqqq pydantic PyYAML
dbutils.library.restartPython()

In [2]:
import sys, os, yaml

print("os.getcwd():", os.getcwd())

for p in sys.path:
    print(p)

os.getcwd(): /Users/s0054120/code-byo/databricks_genai_hackathon/tests
/Users/s0054120/.pyenv/versions/3.11.9/lib/python311.zip
/Users/s0054120/.pyenv/versions/3.11.9/lib/python3.11
/Users/s0054120/.pyenv/versions/3.11.9/lib/python3.11/lib-dynload

/Users/s0054120/code-byo/databricks_genai_hackathon/.venv/lib/python3.11/site-packages
/Users/s0054120/code-byo/databricks_genai_hackathon/.venv/lib/python3.11/site-packages/setuptools/_vendor
/Users/s0054120/code-byo/databricks_genai_hackathon


In [3]:
import sys
import os

sys.path.append(os.getcwd())

from projectroot import add_project_root

project_root_path = add_project_root()
print(project_root_path)

/Users/s0054120/code-byo/databricks_genai_hackathon


In [4]:
with open("../configs/project.yml", "r") as file:
    data = yaml.safe_load(file)

In [5]:
data

{'uc_catalog': 'ds_treaties_model_catalog',
 'uc_schema': 'databricks_genai_hackathon',
 'raw_data_volume': 'raw_data',
 'vector_search_endpoint_name': 'vs_felix_flory',
 'embedding_model_endpoint_name': 'databricks-bge-large-en',
 'secret_scope': 'felix-flory',
 'genie_space_id': '01f00c360aa7147aa93f081d65b4c8e5',
 'mlflow_experiment_base_path': 'Users/felix.flory@rgare.com/mlflow_experiments',
 'mlflow_experiment_name': 'databricks_genai_hackathon',
 'llm_endpoint_names': ['databricks-claude-3-7-sonnet'],
 'vector_search_attributes': {'id_1': {'table_name': 'sec_rag_docs_pages',
   'primary_key': 'chunk_id',
   'embedding_source_column': 'content_chunked',
   'local_clone_path': 'data/sec_rag_docs_pages.snappy.parquet',
   'url': 'https://raw.githubusercontent.com/fflory/databricks_genai_hackathon/main/data/sec_rag_docs_pages.snappy.parquet'}},
 'genie_tables': {'id_1': {'table_name': 'balance_sheet',
   'local_clone_path': 'data/balance_sheet.parquet',
   'url': 'https://raw.github

In [6]:
from configs.project import *

In [7]:
data["mlflow_experiment_base_path"]

'Users/felix.flory@rgare.com/mlflow_experiments'

In [8]:

# print("data:", data)
# print(VectorSearchIndexAttributes(**data['vector_search_attributes']['id_1']).model_dump())
# print(InputTableAttributes(**data['genie_tables']["id_1"]).model_dump())
print(Environment(**data).model_dump())
# print(ProjectConfig(**data).model_dump())

{'uc_catalog': 'ds_treaties_model_catalog', 'uc_schema': 'databricks_genai_hackathon', 'vector_search_endpoint_name': 'vs_felix_flory', 'embedding_model_endpoint_name': 'databricks-bge-large-en', 'raw_data_volume': 'raw_data', 'secret_scope': 'felix-flory', 'genie_space_id': '01f00c360aa7147aa93f081d65b4c8e5', 'llm_endpoint_names': ['databricks-claude-3-7-sonnet'], 'mlflow_experiment_base_path': 'Users/felix.flory@rgare.com/mlflow_experiments', 'mlflow_experiment_name': 'databricks_genai_hackathon'}


In [9]:
# Let's check what fields are actually in the data
print("Keys in data:", list(data.keys()))
print("\nRequired fields for Environment:")
from configs.project import Environment
import inspect
sig = inspect.signature(Environment.__init__)
print([param for param in sig.parameters.keys() if param != 'self'])

Keys in data: ['uc_catalog', 'uc_schema', 'raw_data_volume', 'vector_search_endpoint_name', 'embedding_model_endpoint_name', 'secret_scope', 'genie_space_id', 'mlflow_experiment_base_path', 'mlflow_experiment_name', 'llm_endpoint_names', 'vector_search_attributes', 'genie_tables', 'eval_tables']

Required fields for Environment:
['data']


In [10]:
# Let's look at the current model validator in the Environment class
import inspect
from configs.project import Environment

# Get the source code of the problematic method
print("Current model validator source:")
print(inspect.getsource(Environment.impute_mlflow_experiment_path))

Current model validator source:
    @model_validator(mode="after")
    def impute_mlflow_experiment_path(self) -> "Environment":
        if self.mlflow_experiment_base_path is None:
            from databricks.sdk import WorkspaceClient
            w = WorkspaceClient()
            target_dir = f"/Users/{w.current_user.me().user_name}/mlflow_experiments"
            self.mlflow_experiment_base_path = target_dir
        return self



In [11]:
from configs.project import get_project_config


projectConfig = get_project_config()

In [13]:
# # Restart Python to reload the fixed configs module
# dbutils.library.restartPython()

In [14]:
# Re-add project root and test the fixed configuration
import sys
import os

sys.path.append(os.getcwd())

from projectroot import add_project_root

project_root_path = add_project_root()
print("Project root:", project_root_path)

# Now test the fixed get_project_config function
from configs.project import get_project_config

projectConfig = get_project_config()
print("✅ ProjectConfig created successfully!")
print(f"Catalog: {projectConfig.uc_catalog}")
print(f"Schema: {projectConfig.uc_schema}")
print(f"MLflow base path: {projectConfig.mlflow_experiment_base_path}")

Project root: /Users/s0054120/code-byo/databricks_genai_hackathon
✅ ProjectConfig created successfully!
Catalog: ds_treaties_model_catalog
Schema: databricks_genai_hackathon
MLflow base path: Users/felix.flory@rgare.com/mlflow_experiments


In [ ]:
# projectConfig.model_dump()